# Demo

Run `python -m src.build` first. This notebook just reads the results.

In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd
from IPython.display import Image, display

ROOT = Path("..").resolve()
DB = ROOT / "db" / "signals.db"
FIG = ROOT / "reports" / "figures"

con = sqlite3.connect(DB)
n = pd.read_sql("SELECT COUNT(*) AS n FROM fct_interaction", con).iloc[0, 0]
print(f"rows in fct_interaction: {n:,}")
pd.read_sql("SELECT * FROM fct_interaction LIMIT 5", con)

In [ ]:
switching = pd.read_sql((ROOT / "sql/analysis/01_switching_by_pain_point.sql").read_text(), con)
switching

In [ ]:
for name in [
    "switching_by_pain_point.png",
    "lead_signal_distribution.png",
    "monthly_switching_trend.png",
    "severity_vs_switching.png",
]:
    path = FIG / name
    if path.exists():
        display(Image(filename=str(path)))
    else:
        print(f"missing {name} — run python -m src.insights")

In [ ]:
leads = pd.read_sql((ROOT / "sql/analysis/02_top_lead_signals.sql").read_text(), con)
leads.head(20)

In [ ]:
# one evidence example for a high lead-signal row
if not leads.empty:
    iid = leads.iloc[0]["interaction_id"]
    print(iid)
    display(pd.read_sql(
        "SELECT field_name, rule_id, sentence_no, matched_text "
        "FROM fct_extraction_evidence WHERE interaction_id = ?",
        con,
        params=[iid],
    ))
con.close()